# Debug Eval DAG


Interactive notebook that mirrors the Airflow evaluation DAGs (`dags/eval_dags.py` → `experiments/scripts/eval/runner.py`).
Use this to step through dataset loading, gateway calls, and metric computation without running Airflow.

## 1. Setup Imports and Paths

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent  # experiments/notebooks -> repo root
for p in [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from shared.local_env import load_local_env

load_local_env(repo_root=PROJECT_ROOT)

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

from datasets import load_from_disk

## 2. Load Configuration

Set the same env vars visible inside the Airflow and Jupyter containers.
In Docker Compose, `GATEWAY_URL` is already injected as `http://gateway:9000`; override it only for local or remote runs.
Override `EVAL_BERT_SCORE_MODEL` / `EVAL_SAMPLE_LIMIT` as needed.

In [ ]:
import os

# -- Adjust these to match your environment --
# The setup cell may already have loaded values from the repo-root .env.
# Docker and the bundled Jupyter container already provide
# GATEWAY_URL=http://gateway:9000. setdefault keeps either source
# unless you override it explicitly here.
os.environ.setdefault("GATEWAY_URL", "http://gateway:9000")
os.environ.setdefault("EVAL_SAMPLE_LIMIT", "5")  # small subset for debugging
os.environ.setdefault("EVAL_BERT_SCORE_MODEL", "microsoft/deberta-v3-base")

from shared.config import get_settings

settings = get_settings()
eval_cfg = settings.eval

print(f"Gateway URL   : {settings.gateway.url}")
print(f"BERTScore model: {eval_cfg.metrics.bert_score_model}")
print(f"Sample limit  : {os.environ.get('EVAL_SAMPLE_LIMIT', '(unset)')}")
print(f"Base model    : {settings.gateway.default_model}")

## 3. Choose Eval Parameters

Pick a `(task, dataset, metric)` triple — the same unit the Airflow DAGs run.

In [ ]:
TASK = "chat"
DATASET = "hotpotqa"
METRIC = "bertscore_f1"  # one of: relevance, correctness, bertscore_f1, rouge_l

RAG_ALIAS = "none"
LORA_ALIAS = "none"

# Valid metrics per task (for reference)
TASK_METRICS = {
    "chat": ["relevance", "correctness", "bertscore_f1", "rouge_l"],
    "summarize": ["faithfulness", "coverage", "bertscore_f1", "rouge_l"],
    "code": ["pass_at_1", "executable_rate"],
    "retrieval": ["recall_at_10", "ndcg_at_10"],
}
assert METRIC in TASK_METRICS[TASK], f"Invalid metric {METRIC} for task {TASK}"

## 4. Load Dataset

Load samples from the local Arrow files in `assets/datasets/`.
Same logic as `runner._load_dataset_samples()`.

In [ ]:
ds_dict = load_from_disk(PROJECT_ROOT / "assets" / "datasets" / "hotpotqa")

In [ ]:
samples = []

count = 0
for item in ds_dict["validation"]:
    samples.append({"question": item.get("question", ""), "answer": item.get("answer")})
    count += 1
    if count >= 20:
        break

print(len(samples))

## 5. Fetch Predictions Through the Gateway

This step uses the same SSE helper as `runner.run_eval()`, so the notebook follows the gateway's standard streaming contract and still reconstructs a normal chat-completion shape for inspection.

In [ ]:
from experiments.eval.eval_scripts.runner import _SUITE_KB, _call_gateway
from shared.config import secret_value

predictions: list[str] = []
references: list[str] = []

kb_name = _SUITE_KB.get((TASK, DATASET))
rag_sources = None
if RAG_ALIAS != "none" and kb_name:
    rag_sources = [{"knowledge_base": kb_name, "alias": RAG_ALIAS}]

for i, sample in enumerate(samples):
    question = sample["question"]
    reference = sample.get("answer", "")

    response = _call_gateway(
        messages=[{"role": "user", "content": question}],
        gateway_url=settings.gateway.url,
        rag_sources=rag_sources,
        temperature=eval_cfg.metrics.temperature,
        internal_api_key=secret_value(settings.auth.internal_api_key) or "",
        max_completion_tokens=eval_cfg.metrics.max_completion_tokens,
        expect_rag_context=bool(rag_sources),
    )
    answer = response["choices"][0]["message"]["content"]

    predictions.append(answer)
    references.append(reference)
    print(f"[{i + 1}/{len(samples)}] {question[:80]}...")

print(f"\nGenerated {len(predictions)} predictions")

## 6. Compute Metric

Run the selected metric on the collected predictions/references.
This is the step that OOM'd with `deberta-xlarge-mnli` in Airflow.

In [ ]:
from experiments.eval.eval_scripts.metrics.automatic import compute_bertscore, compute_rouge_l

if METRIC == "bertscore_f1":
    metrics = compute_bertscore(
        predictions,
        references,
        model_name=eval_cfg.metrics.bert_score_model,
    )
elif METRIC == "rouge_l":
    rouge_scores = [
        compute_rouge_l(prediction, reference)
        for prediction, reference in zip(predictions, references, strict=False)
    ]
    metrics = {"rouge_l": sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0.0}
else:
    raise ValueError(
        "This step-by-step cell supports bertscore_f1 and rouge_l. Use runner.run_eval() for judge metrics."
    )

metrics

## 7. Run Full Eval via `runner.run_eval()`

Alternatively, call the runner entry point directly — same function the Airflow DAG invokes.

In [ ]:
from experiments.eval.eval_scripts.runner import run_eval

rows = run_eval(
    task=TASK,
    dataset_name=DATASET,
    metric=METRIC,
    rag_aliases=[RAG_ALIAS],
    lora_aliases=[LORA_ALIAS],
)
rows

## 8. Inspect Results

Show per-sample predictions vs references and the final metric value.

In [ ]:
import pandas as pd

# Per-sample comparison (from the step-by-step path)
df_samples = pd.DataFrame(
    {
        "question": [s["question"] for s in samples],
        "reference": references,
        "prediction": predictions,
    }
)
df_samples

In [ ]:
# Eval-run rows (from the full runner path)
if rows:
    df_results = pd.DataFrame(rows)
    display_cols = [
        "task",
        "dataset_name",
        "metric_name",
        "metric_value",
        "rag_alias",
        "lora_alias",
        "status",
    ]
    df_results[[c for c in display_cols if c in df_results.columns]]